Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

1)Tracking agent behavior with logging, analytics, and debugging.

2)Transforming prompts, tool selection, and output formatting.

3)Adding retries, fallbacks, and early termination logic.

4)Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")


SUMMARIZATION MIDDLEWARE


Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

-Long-running conversations that exceed context windows.

-Multi-turn dialogues with extensive history.

-Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
#message based summarization
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="groq:openai/gpt-oss-120b",
        trigger=("messages",10),
        keep=("messages",4)
    )])

In [3]:
config={"configurable":{"thread_id":"test-1"}}



In [4]:
questions=["what is 2-1?",
"what is 9+0",
"what is 0-0",
"what is 7-2",
"what is 9+7",
"what is 8+7",
"what is 7-1",]
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"Messgaes:{len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='what is 2-1?', additional_kwargs={}, response_metadata={}, id='6a747df9-dafb-4171-b372-6fbad3225a33'), AIMessage(content='2\u202f−\u202f1\u202f=\u202f1.', additional_kwargs={'reasoning_content': 'The user asks a simple math question: "what is 2-1?" The answer is 1. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 78, 'total_tokens': 123, 'completion_time': 0.093449384, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.004033531, 'prompt_tokens_details': None, 'queue_time': 0.375402536, 'total_time': 0.097482915}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04349-1233-73f2-a2b5-e97e1a307618-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 45, 'total_tokens': 123, 'output_token_details':

TOKENSIZE


In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
@tool
def search_hotels(city)->str:
    """search hotels-return long hotels to use more tokens"""
    return f"""hotels in {city}:
    1.Grand hotel-5star,$350 per night,spa,pool,gym
    2.city inn-4star,$180/night,pool
    2.budget stay-3 star,$80/night"""
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='groq:openai/gpt-oss-120b',
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)
config={"configurable":{"thread_id":"test_1"}}
def token_counter(messages):
    total_chars=sum(len(str(m.content)) for m in messages)//4
    return total_chars//4

In [16]:
cities=["paris","london","tokyo"]
for city in cities:
    response=agent.invoke({"messages":[HumanMessage(content=f"find hotels in {city}")]},config=config)
    tokens=token_counter( response["messages"])
    print(f"{city}-{tokens}tokens,{len(response['messages'])} messages")
    print(f"{(response['messages'])}")


paris-61tokens,4 messages
[HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='8e8f9740-c06e-432e-bb81-6f7b4b6f6bde'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to find hotels in Paris. Use the function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_8c473eb4-0aef-40bb-83ec-d8857068b7dc', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 128, 'total_tokens': 176, 'completion_time': 0.101726686, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.005486422, 'prompt_tokens_details': None, 'queue_time': 0.375935364, 'total_time': 0.107213108}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a043b2-ddc3-7cb1-8aa8-4042acd793b

FRACTION

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
@tool
def search_hotels(city)->str:
    """search hotels-return long hotels to use more tokens"""
    return f"""hotels in {city}:
    1.Grand hotel-5star,$350 per night,spa,pool,gym
    2.city inn-4star,$180/night,pool
    2.budget stay-3 star,$80/night"""
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='groq:openai/gpt-oss-120b',
            trigger=("fraction",0.005),
            keep=("fraction",0.002)
        )
    ]
)
config={"configurable":{"thread_id":"test_1"}}
def token_counter(messages):
    total_chars=sum(len(str(m.content)) for m in messages)//4
    return total_chars//4

In [19]:
cities=["paris","london","tokyo"]
for city in cities:
    response=agent.invoke({"messages":[HumanMessage(content=f"find hotels in {city}")]},config=config)
    tokens=token_counter( response["messages"])
    print(f"{city}-{tokens}tokens,{len(response['messages'])} messages")
    print(f"{(response['messages'])}")


paris-327tokens,8 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user wants a list of hotel options in Paris and London, including basic details (price, star rating, key amenities) and guidance on how to refine the search based on preferences such as neighborhood, specific amenities, or budget.\n\n## SUMMARY\n- **Paris hotel search**  \n  - Performed `search_hotels` tool call with `city="Paris"`.  \n  - Received three generic hotel entries:  \n    1. **Grand Hotel** – 5★, $350/night, spa, pool, gym, concierge, central location.  \n    2. **City Inn** – 4★, $180/night, indoor pool, breakfast, Wi‑Fi, near metro.  \n    3. **Budget Stay** – 3★, $80/night, basic rooms, free Wi‑Fi, good transport links.  \n  - Presented a formatted table and highlighted considerations for luxury, family, or budget travelers. Offered to provide further details (address, policies, attractions) or tailor results to specific preferences.\n\n- **London ho

Human In the Loop MiddleWare
+ Code
+ Markdown
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

High-stakes operations requiring human approval (e.g. database writes, financial transactions).

Compliance workflows where human oversight is mandatory.

Long-running conversations where human feedback guides the agent.

In [57]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
@tool
def read_email_tool(email_id:str)->str:
    """mock function to read email by its id"""
    return f"email content for id{email_id}"
@tool
def send_email_tool(recipient:str,subject:str,body:str)->str:
    """mock function to send email"""
    return f"send email to {recipient} with subject{subject}"



In [58]:
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on=
            {
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                
                },
                "read_email_tool":False,
            }
        )
    ]
    
)

In [59]:

config = {"configurable": {"thread_id": "test-approve"}}

result=agent.invoke(
    {
        "messages":[HumanMessage(content="the email sent to harsha@test.con with subject 'hii' and body 'how are you?'")]},config=config
)

In [60]:
result

{'messages': [HumanMessage(content="the email sent to harsha@test.con with subject 'hii' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='13d27a63-6dff-4b8d-adc1-c8e73f1b75f0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to retrieve email with given recipient, subject, and body. Use read_email_tool? It expects email_id. We don\'t have ID. Maybe we can search? There\'s no search tool. Perhaps we need to send email? The user says "the email sent to harsha@test.con with subject \'hii\' and body \'how are you?\'" Could be they want confirmation that such email was sent, or they want to retrieve it. Probably they want to know the email that was sent. We need to retrieve email. But we don\'t have ID. Could we send email? The phrase "the email sent to ..." maybe they want to see the email. Could be they want to retrieve it. Without ID, maybe we can\'t. Could ask for clarification. But likely they want us to send that email. The user says 

In [62]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("@pause approving...")
    result=agent.invoke(
        Command(
            resume={"decisions":[{"type":"approve"}]}
        ),config=config
    )
print(f"@result:{result['messages'][-1].content}")

@pause approving...
@result:Your email has been sent to **harsha@test.con** with the subject **“hii”** and the body:

> how are you?

Let me know if there’s anything else you’d like to do!


In [63]:
result

{'messages': [HumanMessage(content="the email sent to harsha@test.con with subject 'hii' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='13d27a63-6dff-4b8d-adc1-c8e73f1b75f0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to retrieve email with given recipient, subject, and body. Use read_email_tool? It expects email_id. We don\'t have ID. Maybe we can search? There\'s no search tool. Perhaps we need to send email? The user says "the email sent to harsha@test.con with subject \'hii\' and body \'how are you?\'" Could be they want confirmation that such email was sent, or they want to retrieve it. Probably they want to know the email that was sent. We need to retrieve email. But we don\'t have ID. Could we send email? The phrase "the email sent to ..." maybe they want to see the email. Could be they want to retrieve it. Without ID, maybe we can\'t. Could ask for clarification. But likely they want us to send that email. The user says 

In [83]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
@tool
def read_email_tool(email_id:str)->str:
    """mock function to read email by its id"""
    return f"email content for id{email_id}"
@tool
def send_email_tool(recipient:str,subject:str,body:str)->str:
    """mock function to send email"""
    return f"send email to {recipient} with subject{subject}"

agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on=
            {
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                
                },
                "read_email_tool":False,
            }
        )
    ]
    
)

In [ ]:
#reject
config = {"configurable": {"thread_id": "test-reject"}}

result=agent.invoke(
    {
        "messages":[HumanMessage(content="the email sent to harsha@test.con with subject 'hii' and body 'how are you?'")]},config=config
)

In [85]:
result

{'messages': [HumanMessage(content="the email sent to harsha@test.con with subject 'hii' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='77f51d19-9515-45ea-b812-2836d1044dac'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to retrieve the email that was sent to harsha@test.con with subject \'hii\' and body \'how are you?\'. Likely need to search email system. There\'s a function read_email_tool that reads email by email_id, but we don\'t have id. Maybe we need to send email? The user says "the email sent to harsha@test.con with subject \'hii\' and body \'how are you?\'". Possibly they want to retrieve that email? Or they want to confirm it was sent? Could be they want to see the email content. Since we don\'t have ID, maybe we need to ask for clarification? But we could assume we have to send that email. The user statement is ambiguous: "the email sent to harsha@test.con with subject \'hii\' and body \'how are you?\'". Could be they 

In [86]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("@pause approving...")
    result=agent.invoke(
        Command(
            resume={"decisions":[{"type":"reject"}]}
        ),config=config
    )
print(f"@result:{result['messages'][-1].content}")

@pause approving...
@result:I’m not sure what you’d like me to do with that email. Would you like me to:

* Send a new email to **harsha@test.con** with the subject **“hii”** and body **“how are you?”**, or  
* Retrieve or view an existing email that was sent with those details?

Just let me know which action you’d like, and I’ll take care of it!


In [87]:
result

{'messages': [HumanMessage(content="the email sent to harsha@test.con with subject 'hii' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='77f51d19-9515-45ea-b812-2836d1044dac'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to retrieve the email that was sent to harsha@test.con with subject \'hii\' and body \'how are you?\'. Likely need to search email system. There\'s a function read_email_tool that reads email by email_id, but we don\'t have id. Maybe we need to send email? The user says "the email sent to harsha@test.con with subject \'hii\' and body \'how are you?\'". Possibly they want to retrieve that email? Or they want to confirm it was sent? Could be they want to see the email content. Since we don\'t have ID, maybe we need to ask for clarification? But we could assume we have to send that email. The user statement is ambiguous: "the email sent to harsha@test.con with subject \'hii\' and body \'how are you?\'". Could be they 

In [129]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
@tool
def read_email_tool(email_id:str)->str:
    """mock function to read email by its id"""
    return f"email content for id{email_id}"
@tool
def send_email_tool(recipient:str,subject:str,body:str)->str:
    """mock function to send email"""
    return f"send email to {recipient} with subject{subject}"

agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on=
            {
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                
                },
                "read_email_tool":False,
            }
        )
    ]
    
)

In [130]:
#edit
config = {"configurable": {"thread_id": "test-edit"}}

result=agent.invoke(
    {
        "messages":[HumanMessage(content="the email sent to vardhan@test.com with subject 'hii' and body 'how are you?'")]},config=config
)

In [131]:
result

{'messages': [HumanMessage(content="the email sent to vardhan@test.com with subject 'hii' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='93a16212-cb1e-4f16-985b-fbd7ee834635'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user says: "the email sent to vardhan@test.com with subject \'hii\' and body \'how are you?\'" Probably they want to send that email. So we need to use send_email_tool with given fields.', 'tool_calls': [{'id': 'fc_371a0d15-6f76-4f49-8326-2b1c3a6f3ee6', 'function': {'arguments': '{"body":"how are you?","recipient":"vardhan@test.com","subject":"hii"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 175, 'total_tokens': 271, 'completion_time': 0.201124251, 'completion_tokens_details': {'reasoning_tokens': 48}, 'prompt_time': 0.007047084, 'prompt_tokens_details': None, 'queue_time': 0.161831325, 'total_time': 0.208171335}, 'model_name': 'ope

In [132]:
if "__interrupt__" in result:
    print("@paused.... editing.....")
    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":
                        {
                            "name":"send_email_tool",
                            "args":{
                                "recipient":"correct@email.com",
                                "subject":"corrected subject",
                                "body":"this is edited by human before sending" ,
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
print(f"@result:{result['messages'][-1].content}")

@paused.... editing.....
@result:


In [134]:
result

{'messages': [HumanMessage(content="the email sent to vardhan@test.com with subject 'hii' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='93a16212-cb1e-4f16-985b-fbd7ee834635'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user says: "the email sent to vardhan@test.com with subject \'hii\' and body \'how are you?\'" Probably they want to send that email. So we need to use send_email_tool with given fields.', 'tool_calls': [{'id': 'fc_371a0d15-6f76-4f49-8326-2b1c3a6f3ee6', 'function': {'arguments': '{"body":"how are you?","recipient":"vardhan@test.com","subject":"hii"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 175, 'total_tokens': 271, 'completion_time': 0.201124251, 'completion_tokens_details': {'reasoning_tokens': 48}, 'prompt_time': 0.007047084, 'prompt_tokens_details': None, 'queue_time': 0.161831325, 'total_time': 0.208171335}, 'model_name': 'ope